In [ ]:
import time
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP
from cimCommand import CMD,CmdData,Packet
from cimCommand.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar

In [ ]:
chip=CHIP(PS(host="192.168.1.10", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0)

# 1.拟合tg和cond的线性映射

In [ ]:
# 3v,set脉宽1us,reset脉宽1us
tg_map = [1.4,1.5,1.6,1.7,1.8,1.9,2.0,2.1,2.2,2.3,2.4]
cond_map = [190,310,430,510,610,710,850,910,1030,1110,1190]
# 3v,set脉宽1us,reset脉宽10us
# tg_map = [1.4,1.5,1.6,1.7,1.8,1.9,2.0,2.1,2.2,2.3,2.4]
# cond_map = [190,310,430,510,610,710,850,910,1030,1110,1190]

# 
# tg_map = [1.6,1.7,1.8,1.9,2.0,2.1,2.2,2.3]
# cond_map = [550,670,790,890,1010,1090,1170,1250]

tg_map = np.array(tg_map)
cond_map = np.array(cond_map)

coefficients = np.polyfit(tg_map, cond_map, 1)  # 返回斜率和截距
slope = coefficients[0]
intercept = coefficients[1]

y_fit = slope * tg_map + intercept

plt.scatter(tg_map, cond_map, color='blue', label='Data points')  # 原始数据点
plt.plot(tg_map, y_fit, color='red', label=f'cond = {slope:.2f} * tg + {intercept:.2f}')  # 拟合直线
plt.xlabel('tg(v)')
plt.ylabel('cond(us)')
plt.legend()
plt.title('Linear Fit')
plt.show()

# 2.写对应电导值

In [ ]:
def write_verify(write_time,tg_v,target_cond,set_pulse_width,reset_pulse_width,root_path):
    need_read = np.ones((256,256),dtype=bool)
    voltage_base = np.zeros((256,256))
    voltage = np.zeros((256,256))
    cond_sub_base = None
    vmin = min(0,target_cond-250)
    vmax = target_cond+250
    for i in range(write_time):
        print(f"第{i}次写验证")
        voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
        voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
        cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
        np.save(root_path+f"after_write_verify_time={int(i)}.npy", cond_sub_base)

        condition_reset = cond_sub_base > target_cond
        condition_set = cond_sub_base < target_cond
        
        path = root_path+f"{i}.png"
        plot_cond(cond_sub_base,vmin,vmax,title=f"{i},need_reset={int(np.sum(condition_reset))},need_set={int(np.sum(condition_set))}",path = path)

        if i>0:
            if i<5:
                tg_v[condition_reset] -= 0.08
                tg_v[condition_set] += 0.08
            elif i<10:
                tg_v[condition_reset] -= 0.04
                tg_v[condition_set] += 0.04
            else:
                tg_v[condition_reset] -= 0.01
                tg_v[condition_set] += 0.01
        # reset的点
        chip.write_point2(crossbar=condition_reset,write_voltage=2,tg=5,pulse_width=reset_pulse_width,set_device=False)
        chip.write_point2(crossbar=condition_reset,write_voltage=3,tg=tg_v,pulse_width=set_pulse_width,set_device=True)

        # set的点
        chip.write_point2(crossbar=condition_set,write_voltage=3,tg=tg_v,pulse_width=set_pulse_width,set_device=True)


        
    voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
    np.save(root_path+f"after_write_verify_time={write_time}.npy", cond_sub_base)
    np.save(root_path+f"tg_v.npy", tg_v)


    condition_reset = cond_sub_base > target_cond
    condition_set = cond_sub_base < target_cond
    path = root_path+f"{i}.png"
    plot_cond(cond_sub_base,vmin,vmax,title=f"{i},need_reset={int(np.sum(condition_reset))},need_set={int(np.sum(condition_set))}",path = path)
    return cond_sub_base

In [ ]:
for i in range(2,12):
    img_tg = np.ones((256,256))*(i*100 - intercept)/slope
    write_verify(40,img_tg,i*100,1e-6,1e-6,root_path=f"./result/精度/{i*100}us/")

# 3. 验证比特精度

In [ ]:
def good_device_write_verify(write_time,good_device,tg_v,cond_upper,cond_lower,set_pulse_width,reset_pulse_width,root_path):
    need_read = good_device
    voltage_base = np.zeros((256,256))
    voltage = np.zeros((256,256))
    cond_sub_base = None
    vmin = cond_lower
    vmax = cond_upper
    for i in range(write_time):
        print(f"第{i}次写验证")
        voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
        np.save(root_path+f"after_write_verify_time={int(i)}.npy", cond_sub_base)

        condition_reset = (cond_sub_base > cond_upper)&need_read
        condition_set = (cond_sub_base < cond_lower)&need_read
        # need_read = condition_reset|condition_set
        
        path = root_path+f"{i}.png"
        # voltage_base[good_device] = chip.read_point2(crossbar=good_device, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[good_device]
        # voltage[good_device] = chip.read_point2(crossbar=good_device, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[good_device]
        # cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
        plot_cond(cond_sub_base,vmin,vmax,title=f"{i},need_reset={int(np.sum(condition_reset))},need_set={int(np.sum(condition_set))}",path = path)

        if i>0:
            tg_v[condition_reset] -= 0.02
            tg_v[condition_set] += 0.02
        # reset的点
        chip.write_point2(crossbar=condition_reset,write_voltage=3,tg=5,pulse_width=reset_pulse_width,set_device=False)
        chip.write_point2(crossbar=condition_reset,write_voltage=3,tg=tg_v,pulse_width=set_pulse_width,set_device=True)

        # set的点
        chip.write_point2(crossbar=condition_set,write_voltage=3+i*0.025,tg=tg_v,pulse_width=set_pulse_width,set_device=True)

    voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
    np.save(root_path+f"after_write_verify_time={write_time}.npy", cond_sub_base)
    np.save(root_path+f"tg_v.npy", tg_v)
    np.save(root_path+f"good_device.npy", good_device)


    condition_reset = cond_sub_base > cond_upper
    condition_set = cond_sub_base < cond_lower
    path = root_path+f"{i}.png"
    plot_cond(cond_sub_base,vmin,vmax,title=f"{i},need_reset={int(np.sum(condition_reset))},need_set={int(np.sum(condition_set))}",path = path)
    return cond_sub_base

In [ ]:
# voltage_base = chip.read_point2(crossbar=np.ones((256,256)), read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
# voltage = chip.read_point2(crossbar=np.ones((256,256)), read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
# cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)

# np.save("./data/conds.npy",cond_sub_base)

In [64]:
slopes_cond_to_tg = np.load("./data/slopes_cond_to_tg.npy")
intercepts_cond_to_tg = np.load("./data/intercepts_cond_to_tg.npy")
slopes_tg_to_cond = np.load("./data/slopes_tg_to_cond.npy")
intercepts_tg_to_cond = np.load("./data/intercepts_tg_to_cond.npy")
good_device = np.load('./data/good_device.npy')
conds = np.load("./data/conds.npy")
good_device = (slopes_tg_to_cond>800)&(slopes_tg_to_cond<1200)&(conds<200)&good_device
print(np.sum(good_device))

4614


In [ ]:
target = 175
cond_min,cond_max = 200,1000
state_num = 16
threshold = (cond_max-cond_min)/state_num/2
for i in range(15):
    target = target+50
    img_tg = target*slopes_cond_to_tg + intercepts_cond_to_tg
    good_device_write_verify(write_time=30,good_device=good_device,tg_v=img_tg,
                             cond_upper = target+threshold,cond_lower=target-threshold,
                             set_pulse_width = 1e-6,reset_pulse_width=1e-6,root_path=f"./result/5bit精度/{target}us/")

In [ ]:
target = 175
cond_min,cond_max = 200,1000
state_num = 32
threshold = (cond_max-cond_min)/state_num/2
for i in range(15):
    target = target+50
    img_tg = target*slopes_cond_to_tg + intercepts_cond_to_tg
    good_device_write_verify(write_time=30,good_device=good_device,tg_v=img_tg,
                             cond_upper = target+threshold,cond_lower=target-threshold,
                             set_pulse_width = 1e-6,reset_pulse_width=1e-6,root_path=f"./result/6bit精度/{target}us/")

In [ ]:
img_tg = np.ones((256,256))*(10*100 - intercept)/slope-0.1
write_verify(40,img_tg,10*100,1e-6,1e-6,root_path=f"./result/精度/{10*100}us/")

In [ ]:
# def write_verify(write_time,tg_v,target_cond,set_pulse_width,reset_pulse_width,root_path):

In [ ]:
for i in range(2,12):
    img_tg = np.ones((256,256))*(i*100 - intercept)/slope
    write_verify(40,img_tg,i*100,1e-6,1e-6,root_path=f"./result/精度/{i*100}us/")

# 3. 直方图绘制

In [ ]:
target = 225
cond_sub_base_path = []
root_path=f"./result/5bit精度/{target}us/"
for i in range(20):
    cond_sub_base_path.append(root_path+f"after_write_verify_time={int(i)}.npy")
print(cond_sub_base_path)

In [ ]:
interval = 5
bin_edges = np.linspace(0, 2000, int(2000/interval)+1)  
for i,path in enumerate(cond_sub_base_path):
    cond_sub_base = np.load(path)[good_device]
    data = cond_sub_base.flatten()
    counts, bin_edges, _ = plt.hist(data, bins=bin_edges, color='blue', alpha=0.7, edgecolor='black')
    plt.title(f"target = {target},write = {i}")
    plt.xlabel("cond(uS)")
    plt.ylabel("Frequency")
    path = root_path+f"target = {target},write = {i}.png"
    plt.savefig(path)
    plt.show()